# SHAP Local Explanations & Clinical Benchmarking
## CVD Fairness Dissertation — NB2

Purpose: Examine why the model fails for specific female patients using local SHAP
explanations, matched comparisons, and a counterfactual test.

Loads precomputed SHAP arrays from outputs/shap/ — does not recompute them.

Does NOT contain: global summary plots, dependence plots, fairness metrics.
Those are in NB1 and fairness_evaluation.ipynb.

## 1. Imports

In [2]:
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path
from sklearn.preprocessing import StandardScaler

## 2. Paths and Constants

In [3]:
MODEL_PATH  = Path("../models/baseline_models/cardio_xgb_baseline_model.pkl")
TEST_DATA   = Path("../data/test_train_val_sets/cardio_baseline_test.csv")
CONFIG_PATH = Path("../config/dataset_split_sizes.json")
OUTPUT_SHAP = Path("../outputs/shap")

THRESHOLD  = 0.5
GENDER_COL = "gender"
FEMALE_VAL = 0
MALE_VAL   = 1
DROP_COLS  = ["cardio", "stratify"]

## 3. Sanity Checks

Checks that the model, test set, and precomputed SHAP arrays are all consistent
before any analysis runs.

In [4]:
with open(CONFIG_PATH) as f:
    expected_sizes = json.load(f)

test_df = pd.read_csv(TEST_DATA)
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df["cardio"]

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

y_pred = model.predict(X_test)

# Check 1: test set size
assert len(test_df) == expected_sizes["test"], (
    f"Test size mismatch: got {len(test_df)}, expected {expected_sizes['test']}"
)

# Check 2: no target leakage
assert "cardio" not in X_test.columns, "cardio found in X_test"

# Check 3: no nulls
assert X_test.isnull().sum().sum() == 0, "Nulls found in X_test"

# Check 4: class balance
cvd_frac = test_df["cardio"].mean()
assert 0.48 < cvd_frac < 0.52, f"Unexpected class balance: {cvd_frac:.3f}"

# Check 5: sex distribution
female_frac = (test_df[GENDER_COL] == FEMALE_VAL).mean()
assert 0.62 < female_frac < 0.68, f"Unexpected female fraction: {female_frac:.3f}"

print("Test set checks passed")
print(f"  Size      : {len(test_df):,}")
print(f"  Female    : {female_frac:.3f}")
print(f"  CVD rate  : {cvd_frac:.3f}")

Test set checks passed
  Size      : 10,227
  Female    : 0.650
  CVD rate  : 0.495


In [5]:
# Load precomputed SHAP arrays saved by NB1
shap_full   = np.load(OUTPUT_SHAP / "shap_values_baseline_full.npy")
shap_female = np.load(OUTPUT_SHAP / "shap_values_baseline_female.npy")
shap_male   = np.load(OUTPUT_SHAP / "shap_values_baseline_male.npy")
base_value  = float(np.load(OUTPUT_SHAP / "shap_base_value.npy"))

# Check 6: SHAP array dimensions match test set
assert shap_full.shape == X_test.shape, (
    f"Full SHAP shape mismatch: {shap_full.shape} vs {X_test.shape}"
)

female_mask = test_df[GENDER_COL].values == FEMALE_VAL
male_mask   = test_df[GENDER_COL].values == MALE_VAL

X_test_female = X_test[female_mask].copy()
X_test_male   = X_test[male_mask].copy()
y_test_female = y_test.values[female_mask]
y_test_male   = y_test.values[male_mask]
y_pred_female = y_pred[female_mask]
y_pred_male   = y_pred[male_mask]

assert shap_female.shape == X_test_female.shape, (
    f"Female SHAP shape mismatch: {shap_female.shape} vs {X_test_female.shape}"
)
assert shap_male.shape == X_test_male.shape, (
    f"Male SHAP shape mismatch: {shap_male.shape} vs {X_test_male.shape}"
)

# Check 7: subgroup SHAP rows sum to full test set
assert shap_female.shape[0] + shap_male.shape[0] == len(X_test), (
    "Subgroup SHAP row counts do not sum to full test set"
)

print("SHAP array checks passed")
print(f"  Full SHAP   : {shap_full.shape}")
print(f"  Female SHAP : {shap_female.shape}")
print(f"  Male SHAP   : {shap_male.shape}")
print(f"  Base value  : {base_value:.4f}")

SHAP array checks passed
  Full SHAP   : (10227, 11)
  Female SHAP : (6649, 11)
  Male SHAP   : (3578, 11)
  Base value  : -0.0177


C:\Users\kavis\AppData\Local\Temp\ipykernel_19304\448354862.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  base_value  = float(np.load(OUTPUT_SHAP / "shap_base_value.npy"))


## 4. Build Prediction Groups

Defines false negative, true positive, and true negative masks for women and men.
These are used throughout the rest of the notebook.

In [ ]:
fn_mask_f = (y_test_female == 1) & (y_pred_female == 0)
tp_mask_f = (y_test_female == 1) & (y_pred_female == 1)
tn_mask_f = (y_test_female == 0) & (y_pred_female == 0)

fn_mask_m = (y_test_male == 1) & (y_pred_male == 0)
tp_mask_m = (y_test_male == 1) & (y_pred_male == 1)
tn_mask_m = (y_test_male == 0) & (y_pred_male == 0)

fn_women = test_df[female_mask][fn_mask_f].copy()
tp_women = test_df[female_mask][tp_mask_f].copy()
tn_women = test_df[female_mask][tn_mask_f].copy()

fn_men = test_df[male_mask][fn_mask_m].copy()
tp_men = test_df[male_mask][tp_mask_m].copy()
tn_men = test_df[male_mask][tn_mask_m].copy()

# Check masks are mutually exclusive within each sex
assert not (fn_mask_f & tp_mask_f).any(), "FN and TP masks overlap for women"
assert not (fn_mask_m & tp_mask_m).any(), "FN and TP masks overlap for men"

print(f"Women  —  FN: {fn_mask_f.sum()}  TP: {tp_mask_f.sum()}  TN: {tn_mask_f.sum()}")
print(f"Men    —  FN: {fn_mask_m.sum()}  TP: {tp_mask_m.sum()}  TN: {tn_mask_m.sum()}")

## 5. SHAP Comparison: FN vs TP Women

Compares mean SHAP values between false negative and true positive women.
Shows which features are driving the model toward predicting no CVD
for women it misses, compared to women it correctly identifies.

In [ ]:
fn_shap = pd.DataFrame(shap_female[fn_mask_f], columns=X_test.columns)
tp_shap = pd.DataFrame(shap_female[tp_mask_f], columns=X_test.columns)

shap_compare = pd.DataFrame({
    "FN mean SHAP":  fn_shap.mean(),
    "TP mean SHAP":  tp_shap.mean(),
    "difference":    fn_shap.mean() - tp_shap.mean(),
    "FN abs SHAP":   fn_shap.abs().mean(),
    "TP abs SHAP":   tp_shap.abs().mean(),
}).sort_values("difference")

print("Mean SHAP comparison: FN women vs TP women")
print(shap_compare.round(4).to_string())

shap_compare.to_csv(OUTPUT_SHAP / "shap_fn_vs_tp_women_comparison.csv")